# 第 68 课：本地模型的只读评测与身份审计

**从第67课的 Tokenizer / Chat Template 接口，进入真实 checkpoint 迁移前必须做的推理验收。**

本课将模型拆成四项可审计的事实：**读的是哪一份权重、对话前缀是什么、实际输出了哪些 token、究竟因何停止。** 最后才计算严格格式奖励和成功率。

默认实验使用自建 byte-merge tokenizer＋**真正随机初始化的微型因果 Transformer**。它真实进行推理，但没有预训练能力。可选接入**用户已经拥有、可信的本地 Hugging Face Safetensors 检查点**；默认不加载真实 HF 权重、不下载文件、不调用远程代码、不进行训练。

Notebook 和配套 Python 文件**均完全独立**，不需要前几课文件。Python 3.10+，已装 PyTorch；真实本地 HF 模式才需要另行安装兼容 Transformers。

## 0. 这一课怎么运行

在已经装有 PyTorch 的 Python 3 内核中从上到下执行。默认会运行离线审计、固定检查和独立磁盘重载；**只用 CPU，不需要 3060**。

完整脚本三种入口：

```bash
python lesson_68_local_checkpoint_readonly_audit.py --checks-only
python lesson_68_local_checkpoint_readonly_audit.py
python lesson_68_local_checkpoint_readonly_audit.py --inspect-local /path/to/trusted/local/checkpoint
```

如果已经审核本地模型且显式接受加载资源开销，才使用 `--local-model`；它不是默认操作。所有报告输出到全新的时间戳目录，不覆盖已有报告。

**没有预训练模型时不做真实模型推理**，也不制造“实际预训练准确率”。人工正例只是协议夹具，和随机模型生成结果分开存储。

In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""第68课：本地checkpoint只读推理评测、模型身份与停止契约审计。

默认：离线PyTorch随机微型因果模型＋自建byte-merge tokenizer；不下载、不微调。
可选显式本地Transformers+Safetensors模型模式：无远端仓库、不执行远端自定义代码；
真实模型评测只有用户提供受信任本地checkpoint并显式请求时才发生。
本脚本不是沙箱：输入不可信本地模型之前，仍应在受控环境审核文件。
"""
from __future__ import annotations

# %% 01 环境、身份和不可混淆的报告约定
import argparse
import copy
import hashlib
import json
import math
import os
import platform
import random
import re
import struct
import sys
import tempfile
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

try:
    import torch
    from torch import nn
    from torch.nn import functional as F
except ImportError as exc:
    raise SystemExit("需要已安装PyTorch的本地环境；默认不会安装包") from exc

VERSION = "lesson68-readonly-model-audit-v1"
IGNORE = -100
SOURCES = {
 "chat_template":"https://huggingface.co/docs/transformers/chat_templating",
 "generation":"https://huggingface.co/docs/transformers/main_classes/text_generation",
 "safe_load":"https://huggingface.co/docs/transformers/main_classes/model",
 "safetensors":"https://huggingface.co/docs/safetensors/index",
 "pytorch_repro":"https://docs.pytorch.org/docs/stable/notes/randomness.html"
}

class LessonError(ValueError):
    """拒绝含糊的token边界、停止状态、报告格式或资源预算。"""

def require(ok: bool, msg: str) -> None:
    if not bool(ok):
        raise LessonError(msg)

def canonical(value: Any) -> str:
    return json.dumps(value, ensure_ascii=False, sort_keys=True,
                      separators=(",", ":"), allow_nan=False)

def digest(value: Any) -> str:
    return hashlib.sha256(canonical(value).encode("utf-8")).hexdigest()

def write_json(path: Path, value: Any) -> None:
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2, sort_keys=True,
                               allow_nan=False) + "\n", encoding="utf-8")

def bounded_str(value: Any, length: int, label: str) -> str:
    require(type(value) is str and len(value.encode("utf-8")) <= length,
            label + "必须是有界字符串")
    return value

def strict_json(text: str):
    bounded_str(text, 8192, "JSON")
    def pairs(items):
        result = {}
        for key, value in items:
            require(key not in result, "重复JSON字段")
            result[key] = value
        return result
    def bad(_):
        raise LessonError("不接受NaN/Infinity")
    try:
        return json.loads(text, object_pairs_hook=pairs, parse_constant=bad)
    except (json.JSONDecodeError, RecursionError, UnicodeError) as exc:
        raise LessonError("JSON无法严格解析") from exc

def device_from_name(name: str):
    require(name in {"cpu", "cuda", "auto"}, "device应为cpu/cuda/auto")
    if name == "auto":
        name = "cuda" if torch.cuda.is_available() else "cpu"
    require(name != "cuda" or torch.cuda.is_available(), "CUDA不可用，不静默退回CPU")
    return torch.device(name)




torch.set_num_threads(1)
print('Python:',platform.python_version())
print('Torch:',torch.__version__,'CUDA available:',torch.cuda.is_available())

Python: 3.13.5
Torch: 2.10.0+cpu CUDA available: False


## 1. 什么叫“模型身份”？三个等级不能混写

| 等级 | 检查内容 | 能声称什么 | 不能声称什么 |
|---|---|---|---|
| 文件元数据 | 文件名、字节数、修改时间、Safetensors header | 文件结构可见，版本可比对 | 权重全部内容相同 |
| 完整权重 SHA-256 | 逐字节读取每个权重分片 | 已计算的文件内容一致 | 文件来源可信、加载过程绝对安全 |
| 运行张量状态摘要 | 参数名、dtype、shape 与张量字节 | 本次推理前后模型张量不变 | 模型已经学会任务或具备某能力 |

默认本地静态审计只给**元数据身份**，不假装算过所有大权重；`--full-weight-hash` 才完整扫描权重文件，成本与权重文件大小成正比。

讲义会通过一份内容被改动、但文件大小和修改时间保持不变的文件展示：**元数据摘要不一定能发现权重载荷变化，完整哈希可以。**

本课不提供恶意 checkpoint 沙箱；真实目录仍应来自受信任来源。

In [2]:
def tensor_state_hash(state: dict[str,torch.Tensor]) -> str:
    h=hashlib.sha256()
    for name,v in sorted(state.items()):
        require(isinstance(v,torch.Tensor),"状态字典需要张量")
        t=v.detach().cpu().contiguous()
        h.update(canonical([name,list(t.shape),str(t.dtype)]).encode("utf-8"))
        h.update(bytes(t.reshape(-1).view(torch.uint8).tolist()))
    return h.hexdigest()


def stream_sha256(path: Path, chunk_bytes=2**20) -> str:
    require(type(chunk_bytes) is int and chunk_bytes > 0,"hash分块非法")
    h=hashlib.sha256()
    with Path(path).open("rb") as f:
        while True:
            data=f.read(chunk_bytes)
            if not data:break
            h.update(data)
    return h.hexdigest()

## 2. 分词器和 Chat Template 决定你到底问了什么

上一课展示了：普通文本拼接后的编码不一定等于分段编码的 token ID 拼接；用户输入中的字面 `<im_start>` 也不应该自动成为真正的结构 token。

本课复用同样的概念，但实现被复制在本 Notebook，**不是导入第67课脚本**。教学 ByteMergeTokenizer 是模拟组件，不能把它当作 Qwen / Llama / Gemma 的真实 tokenizer。

真正本地 HF 模式直接调用模型自己的 `tokenizer.apply_chat_template(..., tokenize=True, add_generation_prompt=True)`。不会自己猜测角色标记，不重复附加 BOS，也不在已有 assistant 消息后盲目增加 assistant 生成头。

[Hugging Face 官方 Chat Template 指南](https://huggingface.co/docs/transformers/chat_templating)

In [3]:
class ByteMergeChatTokenizer:
    """自建教学分词器：控制token独立于用户文本，普通文本按UTF-8字节合并。

    不假装实现HF的FastTokenizer或任意模型Chat Template。固定控制ID，避免把用户
    输入里的字面控制符误当结构token。真实应用必须使用对应模型的官方tokenizer/template。
    """
    pad_token_id = 0
    bos_token_id = 1
    eos_token_id = 2
    im_start_token_id = 3
    im_end_token_id = 4
    role_ids = {"system":5, "user":6, "assistant":7}
    byte_offset = 16
    merges = (b"OK", b" 42", b"42", b"hello", b"world", b"\n", b"answer", b"JSON", b"{\"answer\":")

    def __init__(self):
        self.merge_ids = {piece: self.byte_offset + 256 + i for i,piece in enumerate(self.merges)}
        self.ids_to_merge = {v:k for k,v in self.merge_ids.items()}
        self.vocab_size = self.byte_offset + 256 + len(self.merges)
        self.specials = {0:"<pad>",1:"<bos>",2:"<eos>",3:"<im_start>",
                         4:"<im_end>",5:"<system>",6:"<user>",7:"<assistant>"}

    def encode(self, text: str, *, add_special_tokens: bool=False) -> list[int]:
        bounded_str(text, 4096, "文本")
        raw = text.encode("utf-8")
        result = []
        index = 0
        while index < len(raw):
            matches = [piece for piece in self.merges if raw.startswith(piece, index)]
            if matches:
                piece = max(matches, key=len)
                result.append(self.merge_ids[piece]);index += len(piece)
            else:
                result.append(self.byte_offset + raw[index]);index += 1
        return ([self.bos_token_id] if add_special_tokens else []) + result

    def decode(self, ids: list[int], *, skip_special_tokens: bool=False) -> str:
        require(type(ids) is list, "token IDs必须是列表")
        raw = bytearray()
        for i in ids:
            require(type(i) is int and 0 <= i < self.vocab_size, "token ID越界")
            if i in self.specials:
                if skip_special_tokens:
                    continue
                raise LessonError("正文decode不允许控制token；传入skip_special_tokens=True")
            if i in self.ids_to_merge: raw.extend(self.ids_to_merge[i])
            elif self.byte_offset <= i < self.byte_offset + 256: raw.append(i - self.byte_offset)
            else: raise LessonError("保留ID不是合法正文token")
        try:
            return bytes(raw).decode("utf-8",errors="strict")
        except UnicodeDecodeError as exc:
            raise LessonError("不完整UTF-8回答；不要用replacement字符伪装成功") from exc

    def apply_chat_template(self, messages: list[dict], *, tokenize: bool=True,
                            add_generation_prompt: bool=False):
        require(type(messages) is list and bool(messages), "messages需要非空列表")
        require(type(add_generation_prompt) is bool, "generation prompt必须是bool")
        ids = [self.bos_token_id]
        for j,m in enumerate(messages):
            require(type(m) is dict and set(m)=={"role","content"}, "message字段必须恰好role/content")
            require(m["role"] in self.role_ids and type(m["content"]) is str, "角色或内容非法")
            require(not (j == len(messages)-1 and add_generation_prompt and m["role"]=="assistant"),
                    "若最后已有assistant，请使用续写语义，不要再添加generation prompt")
            ids.extend([self.im_start_token_id,self.role_ids[m["role"]]])
            ids.extend(self.encode(m["content"],add_special_tokens=False))
            ids.append(self.eos_token_id if m["role"]=="assistant" else self.im_end_token_id)
        if add_generation_prompt:
            ids.extend([self.im_start_token_id,self.role_ids["assistant"]])
        if tokenize:
            return ids
        # 仅用于人类阅读，控制符是标记，不要将其当普通文本用encode()重新编码。
        visible = []
        for i in ids:
            if i in self.specials: visible.append(self.specials[i])
            else: visible.append(self.decode([i],skip_special_tokens=True) if
                                 self._token_is_full_utf8(i) else "<byte-part>")
        return "".join(visible)

    def _token_is_full_utf8(self, i):
        if i in self.ids_to_merge:
            try: self.ids_to_merge[i].decode("utf-8");return True
            except UnicodeError: return False
        return self.byte_offset <= i < self.byte_offset + 128

    def contract(self) -> dict:
        return {"type":"toy-byte-merge", "specials":self.specials,
                "merges":[p.hex() for p in self.merges],"vocab":self.vocab_size,
                "prefix":"BOS+message start/role/body+end+assistant header",
                "eos":self.eos_token_id,"pad":self.pad_token_id}


def messages_for_task(question: str) -> list[dict]:
    bounded_str(question, 1500, "question")
    require(bool(question.strip()), "问题不能为空")
    return [{"role":"system","content":"Return only JSON with answer as a string."},
            {"role":"user","content":question}]

In [4]:
tok = ByteMergeChatTokenizer()
print('词表大小:', tok.vocab_size)
print('OK 一次编码:', tok.encode('OK'))
print('O/K 分段编码:', tok.encode('O')+tok.encode('K'))
print('用户字面控制标记是普通文本:', tok.encode('<im_start>'))
prefix = tok.apply_chat_template(messages_for_task('Calculate 2 plus 3.'),
                                  tokenize=True, add_generation_prompt=True)
print('前缀长度:',len(prefix),'assistant头:',prefix[-2:])

词表大小: 281
OK 一次编码: [272]
O/K 分段编码: [95, 91]
用户字面控制标记是普通文本: [76, 121, 125, 111, 131, 132, 113, 130, 132, 78]
前缀长度: 61 assistant头: [3, 7]


## 3. 真正运行因果注意力，但不能把随机权重叫成真实 LLM

模型是两层 Pre-LN Transformer，隐藏维度 32、四个注意力头，使用严格因果掩码；没有 KV cache、量化和 LoRA 更新。这里只做只读推理，不回传梯度。

教学模型来自确定性的随机初始化，然后保存到文件，再从**新建模型对象**加载；这样可以验证“检查点身份 → 推理”完整链路。

**随机模型成绩很可能是 0。** 这是诚实的能力边界，而不是需要偷偷替换答案或增加针对测试集的训练。

In [5]:
@dataclass(frozen=True)
class TinyConfig:
    vocab_size: int
    hidden: int=32
    heads: int=4
    layers: int=2
    max_context: int=256

    def validate(self):
        require(all(type(x) is int and x>0 for x in asdict(self).values()),"模型配置应为正整数")
        require(self.hidden % self.heads ==0 and self.max_context>=32,"注意力头或上下文配置不合法")


class CausalBlock(nn.Module):
    def __init__(self, cfg: TinyConfig):
        super().__init__()
        self.ln1=nn.LayerNorm(cfg.hidden)
        self.attn=nn.MultiheadAttention(cfg.hidden,cfg.heads,batch_first=True,dropout=0.0)
        self.ln2=nn.LayerNorm(cfg.hidden)
        self.ff=nn.Sequential(nn.Linear(cfg.hidden,4*cfg.hidden),nn.GELU(),
                              nn.Linear(4*cfg.hidden,cfg.hidden))

    def forward(self,x,pad_mask,causal_mask):
        y,_=self.attn(self.ln1(x),self.ln1(x),self.ln1(x),
                      key_padding_mask=pad_mask,attn_mask=causal_mask,need_weights=False)
        x=x+y
        return x+self.ff(self.ln2(x))


class TinyCausalLM(nn.Module):
    def __init__(self,cfg: TinyConfig):
        super().__init__();cfg.validate();self.cfg=cfg
        self.token=nn.Embedding(cfg.vocab_size,cfg.hidden)
        self.position=nn.Embedding(cfg.max_context,cfg.hidden)
        self.blocks=nn.ModuleList([CausalBlock(cfg) for _ in range(cfg.layers)])
        self.norm=nn.LayerNorm(cfg.hidden)
        self.head=nn.Linear(cfg.hidden,cfg.vocab_size,bias=False)

    def forward(self,input_ids: torch.Tensor,attention_mask: torch.Tensor) -> torch.Tensor:
        require(input_ids.ndim==2 and input_ids.dtype==torch.long,"input_ids需要[B,T] long")
        b,t=input_ids.shape
        require(b>0 and 0<t<=self.cfg.max_context,"模型上下文超限")
        require(attention_mask.shape==input_ids.shape and attention_mask.dtype==torch.bool,
                "attention_mask形状/type错误")
        require(bool(attention_mask[:,0].all()),"课堂训练只支持右填充，拒绝左填充")
        require(not bool(((~attention_mask[:,:-1]) & attention_mask[:,1:]).any()),"不支持中间空洞")
        require(bool(((input_ids>=0)&(input_ids<self.cfg.vocab_size)).all()),"输入ID越界")
        arange=torch.arange(t,device=input_ids.device)
        x=self.token(input_ids)+self.position(arange)[None]
        causal_mask=torch.ones(t,t,dtype=torch.bool,device=input_ids.device).triu(1)
        for block in self.blocks:
            x=block(x,~attention_mask,causal_mask)
        return self.head(self.norm(x))


def normalize_stop_ids(eos_spec: int|list[int]|tuple[int,...]|None,
                       vocab_size: int|None=None) -> list[int]:
    if type(eos_spec) is int: values=[eos_spec]
    elif type(eos_spec) in (list,tuple) and bool(eos_spec): values=list(eos_spec)
    else: raise LessonError("未配置EOS/EOT时不能假定任何token是结束符")
    require(all(type(v) is int and v >=0 and (vocab_size is None or v < vocab_size)
                for v in values) and len(values)==len(set(values)),"停止ID重复或超出词表")
    return values

In [6]:
model_cfg = TinyConfig(vocab_size=tok.vocab_size)
print('toy模型配置:', asdict(model_cfg))
print('参数量:',sum(p.numel() for p in TinyCausalLM(model_cfg).parameters()))

toy模型配置: {'vocab_size': 281, 'hidden': 32, 'heads': 4, 'layers': 2, 'max_context': 256}
参数量: 51648


## 4. 固定评测题与严谨分母

题目是自定义的八条算术 JSON 任务，例如“2 加 3”，期望正文是：

```json
{"answer":"5"}
```

任务判定必须满足：**正常出现结束 token、正文是严格 JSON 对象、恰好一个 `answer` 字段且为字符串、值匹配。** 字符串 "5" 和数字 5 不是同一格式。

真实模型评测结果应同时给出：总题数、合格完成数、未知/中断数、成功数、各种停止原因。中断样本不能直接算成“模型错误”的 0 奖励。

本课只做固定小题协议验收，不称为经过统计设计的公开能力基准。

In [7]:
def fixed_cases() -> list[dict]:
    """人工的可计算小题；用于审计而非权威能力基准。"""
    specifications = [
        ("a01", "Calculate 2 plus 3. Reply in the required JSON format.", "5"),
        ("a02", "Calculate 7 minus 2. Reply in the required JSON format.", "5"),
        ("a03", "Calculate 6 times 2. Reply in the required JSON format.", "12"),
        ("a04", "Calculate 10 plus 4. Reply in the required JSON format.", "14"),
        ("a05", "Calculate 11 minus 3. Reply in the required JSON format.", "8"),
        ("a06", "Calculate 3 times 4. Reply in the required JSON format.", "12"),
        ("a07", "Calculate 9 plus 8. Reply in the required JSON format.", "17"),
        ("a08", "Calculate 10 minus 9. Reply in the required JSON format.", "1"),
    ]
    cases = [{"id": i, "question": q, "expected": a} for i,q,a in specifications]
    validate_cases(cases)
    return cases


def validate_cases(cases: list[dict]) -> None:
    require(type(cases) is list and bool(cases) and len(cases) <= 100, "题目数量需要1..100")
    names = set()
    for c in cases:
        require(type(c) is dict and set(c)=={"id","question","expected"}, "题目字段不符合契约")
        require(type(c["id"]) is str and re.fullmatch(r"[a-z][0-9]{2}",c["id"]),"题号无效")
        require(c["id"] not in names,"题号重复")
        names.add(c["id"])
        bounded_str(c["question"],1000,"题目文本")
        require(bool(c["question"]),"题目不能为空")
        require(type(c["expected"]) is str and re.fullmatch(r"[0-9]+",c["expected"]) is not None,"期望必须为非负整数字符串")

In [8]:
CASES = fixed_cases()
print('数据集:',len(CASES),'条；数据sha256:',digest(CASES))
for item in CASES[:3]:
    print(item)

数据集: 8 条；数据sha256: 2042390a662128d06be58ab8302487c5ad63711997bd7d196958c2fdc847583b
{'id': 'a01', 'question': 'Calculate 2 plus 3. Reply in the required JSON format.', 'expected': '5'}
{'id': 'a02', 'question': 'Calculate 7 minus 2. Reply in the required JSON format.', 'expected': '5'}
{'id': 'a03', 'question': 'Calculate 6 times 2. Reply in the required JSON format.', 'expected': '12'}


## 5. 停止状态必须从 token ID 判定，不能猜

- `eos`：序列包含配置中的 EOS/EOT，且位置和后续 padding 合法。
- `budget_cap`：生成正好用完预算但没有结束符。它是任务预先定义的合法终止，可判错。
- `interrupted`：预算没用完且也没有 EOS，或者明确发生基础设施中断；**结果未知**，不能充作失败样本。
- `ambiguous_stop_padding`：EOS 与 PAD 使用相同 ID，后面又有多个相同 ID，无法仅凭 IDs 确认哪些是模型停止、哪些是填充。

如果 EOS 恰好发生在最后一个预算 token，状态仍然是 **eos**，不能把所有“长度=预算”都当 cap。模型可能配置**多个 EOS/EOT ID**，必须一并处理。停止之后再出现非 PAD token，属于生成契约异常。

[Hugging Face GenerationConfig：max_new_tokens、多个 EOS 和返回序列](https://huggingface.co/docs/transformers/main_classes/text_generation)

In [9]:
def audit_generated_tokens(generated_ids: list[int], eos_ids: list[int], max_new_tokens: int,
                           *, pad_id: int|None=None, interrupted: bool=False) -> dict:
    require(type(max_new_tokens) is int and 1 <= max_new_tokens <= 512,"生成预算非法")
    eos_ids=normalize_stop_ids(eos_ids)
    require(type(generated_ids) is list and all(type(x) is int and x>=0 for x in generated_ids),
            "生成序列必须是非负整数token IDs")
    require(len(generated_ids) <= max_new_tokens,"生成长度超过预算")
    require(pad_id is None or (type(pad_id) is int and pad_id>=0),"pad_id非法")
    eos_positions=[j for j,x in enumerate(generated_ids) if x in eos_ids]
    if interrupted:
        return {"state":"interrupted","eligible":False,"body_ids":generated_ids[:],
                "effective_length":len(generated_ids),"observed_tokens":len(generated_ids),
                "stop_id":None,"reason":"infrastructure_interruption_not_zero_reward"}
    if eos_positions:
        first=eos_positions[0]; trailing=generated_ids[first+1:]
        if trailing:
            require(pad_id is not None and all(x==pad_id for x in trailing),
                    "停止符后出现非padding token；生成契约损坏")
            if pad_id in eos_ids:
                return {"state":"ambiguous_stop_padding","eligible":False,"body_ids":generated_ids[:first],
                        "effective_length":first+1,"observed_tokens":len(generated_ids),
                        "stop_id":generated_ids[first],"reason":"pad_and_eos_alias_with_trailing_tokens"}
        return {"state":"eos","eligible":True,"body_ids":generated_ids[:first],
                "effective_length":first+1,"observed_tokens":len(generated_ids),
                "stop_id":generated_ids[first],"reason":"normal_stop"}
    if len(generated_ids)==max_new_tokens:
        return {"state":"budget_cap","eligible":True,"body_ids":generated_ids[:],
                "effective_length":len(generated_ids),"observed_tokens":len(generated_ids),
                "stop_id":None,"reason":"task_length_budget"}
    return {"state":"interrupted","eligible":False,"body_ids":generated_ids[:],
            "effective_length":len(generated_ids),"observed_tokens":len(generated_ids),
            "stop_id":None,"reason":"early_return_without_stop_id"}

In [10]:
eos_ids = [2,4]
examples = [
 ('正常EOS',[17,2],3,None),
 ('恰好预算处EOS',[17,18,2],3,None),
 ('正常cap',[17,18,19],3,None),
 ('提前返回无EOS',[17,18],3,None),
 ('EOS/PAD别名后面还有token',[17,2,2],3,2),
]
for name, ids, limit, pad in examples:
    result = audit_generated_tokens(ids,eos_ids,limit,pad_id=pad)
    print(name, result['state'], 'eligible=',result['eligible'])

正常EOS eos eligible= True
恰好预算处EOS eos eligible= True
正常cap budget_cap eligible= True
提前返回无EOS interrupted eligible= False
EOS/PAD别名后面还有token ambiguous_stop_padding eligible= False


## 6. 解码和严格评分：不能吞掉错误

Byte-level tokenizer 可能输出不完整 UTF-8；不能用替换字符悄悄修复，导致一个原本无法解码的字符串误判为合法。生成 ID 必须保留，正文解码失败时如实记录 `decode_error`。

判定统一使用严格 JSON 解析：拒绝重复字段、`NaN`、多余字段、错误字段类型和错误值。

完成 JSON 但没有生成 EOS，依然是本任务的 **0**；这是“输出格式+停止协议”的验收，而不等同于只测试模型算术知识。

In [11]:
def decode_or_reason(tokenizer, ids: list[int]) -> tuple[str|None,str|None]:
    """不使用replacement字符修复不完整UTF-8；输出错误仍保留真实token ID。"""
    try:
        return tokenizer.decode(ids,skip_special_tokens=False),None
    except (UnicodeError,LessonError,ValueError) as exc:
        return None,type(exc).__name__


def grade_strict_json(body: str|None, state: str, expected: str,
                      *, decode_error: str|None=None) -> dict:
    require(type(expected) is str and re.fullmatch(r"[0-9]+",expected) is not None,"期望答案非法")
    require(state in {"eos","budget_cap","interrupted","ambiguous_stop_padding"},
            "停止状态非法")
    if state in {"interrupted","ambiguous_stop_padding"}:
        return {"reward":None,"eligible":False,"reason":state}
    if state=="budget_cap":
        return {"reward":0.0,"eligible":True,"reason":"missing_eos"}
    if decode_error is not None:
        return {"reward":0.0,"eligible":True,"reason":"decode_error"}
    require(type(body) is str,"eos状态必须具有可读正文或解码错误标记")
    try:
        obj=strict_json(body)
        require(type(obj) is dict and set(obj)=={"answer"} and type(obj["answer"]) is str,
                "必须恰有字符串answer字段")
    except LessonError:
        return {"reward":0.0,"eligible":True,"reason":"invalid_schema"}
    success=obj["answer"]==expected
    return {"reward":float(success),"eligible":True,
            "reason":"correct" if success else "wrong_answer"}


def grade_generation(tokenizer, generated_ids: list[int], eos_ids: list[int], budget: int,
                     expected: str, *, pad_id: int|None=None,
                     interrupted: bool=False) -> dict:
    audit=audit_generated_tokens(generated_ids,eos_ids,budget,pad_id=pad_id,interrupted=interrupted)
    text,err=decode_or_reason(tokenizer,audit["body_ids"])
    grade=grade_strict_json(text,audit["state"],expected,decode_error=err)
    return {"audit":audit,"body":text,"decode_error":err,"grade":grade}


def summarize_results(rows: list[dict]) -> dict:
    require(type(rows) is list and bool(rows),"没有评测记录")
    eligible=[r for r in rows if r["grade"]["eligible"]]
    successes=sum(r["grade"]["reward"] for r in eligible)
    states={key:sum(r["audit"]["state"]==key for r in rows)
            for key in ("eos","budget_cap","interrupted","ambiguous_stop_padding")}
    by_reason={key:sum(r["grade"]["reason"]==key for r in rows)
               for key in sorted({r["grade"]["reason"] for r in rows})}
    return {"total":len(rows),"eligible":len(eligible),"unknown":len(rows)-len(eligible),
            "successes":int(successes),"strict_success_over_eligible":None if not eligible else successes/len(eligible),
            "strict_success_over_all":successes/len(rows),
            "states":states,"grade_reasons":by_reason,
            "denominator_policy":"unknown excludes infrastructure interruption and pad/EOS ambiguity; also report over_all"}

In [12]:
for body,state in [('{"answer":"5"}','eos'),('{"answer":5}','eos'),
                   ('{"answer":"5","other":0}','eos'),('{"answer":"5"}','budget_cap')]:
    print(repr(body),state,grade_strict_json(body,state,'5'))
print('基础设施中断:',grade_strict_json(None,'interrupted','5'))

'{"answer":"5"}' eos {'reward': 1.0, 'eligible': True, 'reason': 'correct'}
'{"answer":5}' eos {'reward': 0.0, 'eligible': True, 'reason': 'invalid_schema'}
'{"answer":"5","other":0}' eos {'reward': 0.0, 'eligible': True, 'reason': 'invalid_schema'}
'{"answer":"5"}' budget_cap {'reward': 0.0, 'eligible': True, 'reason': 'missing_eos'}
基础设施中断: {'reward': None, 'eligible': False, 'reason': 'interrupted'}


## 7. 正例夹具不能冒充模型能力

下面的人为设定输出，是为了覆盖真实模型很可能无法稳定命中的测试边界：正确 JSON＋EOS、错误值、错误 schema、预算结束、提前返回、明确中断、无效 UTF-8。

这些数据属于 `scripted_test_fixture_not_a_language_model`，**不是任何模型生成的成绩**。固定脚本与随机模型的报告分别保存，避免混用分母。

In [13]:
def scripted_fixture_results() -> dict:
    tok=ByteMergeChatTokenizer()
    expected="5";correct='{"answer":"5"}';wrong='{"answer":"6"}'
    eos=tok.eos_token_id;pad=tok.pad_token_id
    fixtures=[
      ("correct_eos",tok.encode(correct)+[eos],len(tok.encode(correct))+2,False),
      ("wrong_eos",tok.encode(wrong)+[eos],len(tok.encode(wrong))+1,False),
      ("schema_eos",tok.encode('{"answer":5}')+[eos],len(tok.encode('{"answer":5}'))+1,False),
      ("budget_cap",tok.encode(correct),len(tok.encode(correct)),False),
      ("interrupted_early",tok.encode('x'),len(tok.encode('x'))+4,False),
      ("explicit_interrupted",tok.encode('x')+[eos],len(tok.encode('x'))+1,True),
      ("invalid_utf8_eos",[tok.byte_offset+0xE4,eos],3,False),
    ]
    rows=[]
    for name,ids,budget,interrupted in fixtures:
        result=grade_generation(tok,ids,[eos],budget,expected,pad_id=pad,interrupted=interrupted)
        rows.append({"fixture":name,"generated_ids":ids,**result})
    return {"scope":"scripted_test_fixture_not_a_language_model","summary":summarize_results(rows),
            "rows":rows}

In [14]:
FIXTURES = scripted_fixture_results()
print('夹具摘要:', FIXTURES['summary'])
for row in FIXTURES['rows']:
    print(row['fixture'],row['audit']['state'],row['grade'])

夹具摘要: {'total': 7, 'eligible': 5, 'unknown': 2, 'successes': 1, 'strict_success_over_eligible': 0.2, 'strict_success_over_all': 0.14285714285714285, 'states': {'eos': 4, 'budget_cap': 1, 'interrupted': 2, 'ambiguous_stop_padding': 0}, 'grade_reasons': {'correct': 1, 'decode_error': 1, 'interrupted': 2, 'invalid_schema': 1, 'missing_eos': 1, 'wrong_answer': 1}, 'denominator_policy': 'unknown excludes infrastructure interruption and pad/EOS ambiguity; also report over_all'}
correct_eos eos {'reward': 1.0, 'eligible': True, 'reason': 'correct'}
wrong_eos eos {'reward': 0.0, 'eligible': True, 'reason': 'wrong_answer'}
schema_eos eos {'reward': 0.0, 'eligible': True, 'reason': 'invalid_schema'}
budget_cap budget_cap {'reward': 0.0, 'eligible': True, 'reason': 'missing_eos'}
interrupted_early interrupted {'reward': None, 'eligible': False, 'reason': 'interrupted'}
explicit_interrupted interrupted {'reward': None, 'eligible': False, 'reason': 'interrupted'}
invalid_utf8_eos eos {'reward': 0.0

## 8. 从磁盘加载 toy checkpoint，审计前后权重

这一步真实写出 `toy_random_model.pt`，随后通过 `torch.load(...,weights_only=True)` 加载本课自己写的可信文件，验证 tokenizer 契约与参数内容摘要，再将所有参数冻结。

**不用于未知第三方 `.pt/.bin` 权重。** 可选真实模型模式只接受本地 Safetensors，绝不退回使用 pickle 文件。

默认数据输出创建新的时间戳目录；模型文件不会被覆盖。推理前后检查运行参数摘要与磁盘文件 SHA-256。

In [15]:
def toy_checkpoint_path(directory: Path) -> Path:
    return Path(directory)/"toy_random_model.pt"


def create_toy_checkpoint(directory: Path, seed: int=6801) -> dict:
    require(type(seed) is int and seed>=0,"toy种子非法")
    directory=Path(directory)
    directory.mkdir(parents=True,exist_ok=True)
    target=toy_checkpoint_path(directory)
    require(not target.exists(),"不覆盖现有toy检查点")
    rng_before=torch.random.get_rng_state().clone()
    torch.manual_seed(seed)
    tok=ByteMergeChatTokenizer()
    model=TinyCausalLM(TinyConfig(vocab_size=tok.vocab_size))
    state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
    torch.random.set_rng_state(rng_before)
    payload={"version":VERSION,"scope":"random_toy_not_pretrained",
             "config":asdict(model.cfg),"tokenizer_contract":tok.contract(),
             "state_hash":tensor_state_hash(state),"state":state}
    torch.save(payload,target)
    return {"filename":target.name,"file_sha256":stream_sha256(target),
            "state_hash":payload["state_hash"],"parameter_count":sum(p.numel() for p in model.parameters()),
            "seed":seed,"file_size":target.stat().st_size,
            "identity_level":"exact_toy_tensor_state_plus_file_sha256"}


def load_toy_checkpoint(path: Path,device: str="cpu") -> tuple[TinyCausalLM,ByteMergeChatTokenizer,dict]:
    p=Path(path)
    require(p.is_file() and not p.is_symlink(),"必须提供本课自己生成的普通toy模型文件")
    data=torch.load(p,map_location="cpu",weights_only=True)
    tok=ByteMergeChatTokenizer()
    require(data.get("version")==VERSION and data.get("scope")=="random_toy_not_pretrained",
            "toy检查点版本/类型不一致")
    require(data.get("tokenizer_contract")==tok.contract(),"toy tokenizer不匹配")
    require(tensor_state_hash(data["state"])==data["state_hash"],"toy模型内容摘要不匹配")
    cfg=TinyConfig(**data["config"])
    require(cfg.vocab_size==tok.vocab_size,"词表与模型维度不匹配")
    model=TinyCausalLM(cfg)
    model.load_state_dict(data["state"],strict=True)
    model.requires_grad_(False).eval().to(device_from_name(device))
    return model,tok,{"scope":"toy_random_not_pretrained", "state_hash":data["state_hash"],
                      "checkpoint_file_sha256":stream_sha256(p),"model_parameters":sum(x.numel() for x in model.parameters()),
                      "model_config":asdict(cfg)}

### 推理用真实 logits，不用参考答案替换生成输出

toy 贪心、固定随机采样都在每一步前向产生 logits。只允许普通 token 和 EOS；停止与打分统一由上一节的审计函数处理。模型不会访问期望答案来决定输出。

默认 toy 最多生成 32 个新 token；在 CPU 可直接运行。结果可能没有 EOS，正确标为预算终止。

In [16]:
@torch.no_grad()
def generate_toy_ids(model: TinyCausalLM,prefix: list[int],tok: ByteMergeChatTokenizer,
                     limit: int=32, *,greedy: bool=True, seed: int=68) -> list[int]:
    require(type(limit) is int and 1<=limit<=64,"toy生成预算非法")
    require(type(prefix) is list and bool(prefix) and all(type(x) is int and 0<=x<tok.vocab_size for x in prefix),
            "toy输入必须是合法token列表")
    require(len(prefix)+limit<=model.cfg.max_context,"输入加新生成预算超过toy上下文")
    old_mode=model.training
    model.eval()
    dev=next(model.parameters()).device
    generator=torch.Generator(device=dev).manual_seed(seed)
    result=[]
    try:
        for _ in range(limit):
            ids=torch.tensor([prefix+result],dtype=torch.long,device=dev)
            logits=model(ids,torch.ones_like(ids,dtype=torch.bool))[0,-1].clone()
            eos=logits[tok.eos_token_id].clone()
            logits[:tok.byte_offset]=-torch.inf
            logits[tok.eos_token_id]=eos
            probs=F.softmax(logits,dim=-1)
            require(bool(torch.isfinite(probs).all()) and float(probs.sum())>0,"toy采样概率非法")
            chosen=int(probs.argmax()) if greedy else int(torch.multinomial(probs,1,generator=generator))
            result.append(chosen)
            if chosen==tok.eos_token_id:break
    finally:
        model.train(old_mode)
    return result


def evaluate_toy_readonly(model: TinyCausalLM,tok: ByteMergeChatTokenizer,cases: list[dict],
                          max_new_tokens=32,seed=6811,greedy=True) -> dict:
    validate_cases(cases)
    before=tensor_state_hash(model.state_dict())
    old_mode=model.training
    records=[]
    for j,case in enumerate(cases):
        prompt=tok.apply_chat_template(messages_for_task(case["question"]),
                                       tokenize=True,add_generation_prompt=True)
        require(prompt[-2:]==[tok.im_start_token_id,tok.role_ids["assistant"]],
                "没有assistant生成头")
        generated=generate_toy_ids(model,prompt,tok,max_new_tokens,greedy=greedy,seed=seed+j)
        scored=grade_generation(tok,generated,[tok.eos_token_id],max_new_tokens,case["expected"],
                                pad_id=tok.pad_token_id)
        records.append({"id":case["id"],"expected":case["expected"],
                        "prompt_digest":digest(prompt),"prompt_token_count":len(prompt),
                        "generated_token_ids":generated,"generated_token_count":len(generated),
                        "generation_mode":"greedy" if greedy else "sampled_fixed_seed",
                        **scored})
    after=tensor_state_hash(model.state_dict())
    require(before==after,"只读评测修改了模型权重")
    require(model.training==old_mode,"评测意外修改train/eval状态")
    require(all(p.grad is None for p in model.parameters()),"只读评测不应产生梯度")
    return {"summary":summarize_results(records),"rows":records,
            "before_hash":before,"after_hash":after,"all_parameters_frozen":True,
            "mode":"toy_offline_random_causal_model_not_pretrained"}

In [17]:
with tempfile.TemporaryDirectory() as tmp:
    info=create_toy_checkpoint(Path(tmp),seed=6801)
    toy, toy_tok, identity=load_toy_checkpoint(Path(tmp)/'toy_random_model.pt')
    probe=evaluate_toy_readonly(toy,toy_tok,CASES[:2],max_new_tokens=8,greedy=True)
    print('模型身份:', identity['scope'], identity['state_hash'][:16])
    print('推理前后hash完全一致:',probe['before_hash']==probe['after_hash'])
    print('两题实际toy输出:',[(r['id'],r['audit']['state'],r['grade']['reward']) for r in probe['rows']])

模型身份: toy_random_not_pretrained 77b4974834612149
推理前后hash完全一致: True
两题实际toy输出: [('a01', 'budget_cap', 0.0), ('a02', 'budget_cap', 0.0)]


## 9. 静态审计受信任本地 Safetensors：不需要安装 Transformers

只检查本地路径包含 `config.json` 和一个或多个 `*.safetensors`；拒绝符号链接、缺失分片索引、pickle-only、跨目录 shard 路径、损坏或越界的 Safetensors header。

静态审计会读取模型配置、分片文件大小及必要的 Safetensors header。对多分片情况，还核对索引里每个 tensor 的文件归属。

**资源闸门：** 默认仅允许总权重文件字节数不超过 2 GiB，超出即失败，不尝试加载。你可以明确修改阈值，但这**不是显存足够的保证**，因为还有 KV cache、激活、加载副本、框架和系统开销。

如果不加 `--full-weight-hash`，得到的是元数据/文件头指纹，**不是完整权重的加密摘要**。

In [18]:
def ensure_local_directory(path: str|Path) -> Path:
    raw=Path(path).expanduser()
    require(not raw.is_symlink(),"不能通过符号链接提供模型目录")
    require(raw.is_dir() and (raw/"config.json").is_file(),
            "必须是包含config.json的现有本地目录，不接受仓库ID或URL")
    folder=raw.resolve(strict=True)
    require(not folder.is_symlink(),"真实目录不能是符号链接")
    return folder


def read_limited_json(path: Path,max_bytes: int=2_000_000):
    require(path.is_file() and not path.is_symlink(),"JSON文件必须是普通文件，不允许符号链接")
    require(0<path.stat().st_size<=max_bytes,"JSON文件为空或超过安全读取上限")
    return strict_json(path.read_text(encoding="utf-8"))


def safetensors_header(path: Path) -> dict:
    """只读前缀，不通过torch.load打开权重；不是对任意恶意文件的安全认证。"""
    require(path.is_file() and not path.is_symlink(),"权重文件不能是符号链接")
    size=path.stat().st_size
    require(size>=16,"safetensors文件太短")
    with path.open("rb") as source:
        first=source.read(8)
        require(len(first)==8,"权重文件缺失header长度")
        n=struct.unpack("<Q",first)[0]
        require(2<=n<=16*2**20 and n+8<=size,"safetensors header非法或过大")
        head=source.read(n)
        require(len(head)==n,"safetensors header截断")
    header=strict_json(head.decode("utf-8")) if n<=8192 else json.loads(head.decode("utf-8"),
        object_pairs_hook=lambda pairs: _no_duplicate_object(pairs),
        parse_constant=lambda x: (_ for _ in ()).throw(LessonError("非法常量")))
    require(type(header) is dict and bool(header),"safetensors需要JSON对象header")
    data_len=size-8-n
    names=[]
    for key,value in header.items():
        if key=="__metadata__":
            require(type(value) is dict,"safetensors metadata需要对象")
            continue
        require(type(key) is str and 0<len(key)<=4096 and type(value) is dict,
                "safetensors tensor条目非法")
        require(type(value.get("dtype")) is str and type(value.get("shape")) is list,
                "safetensors dtype/shape缺失")
        require(all(type(dim) is int and dim>=0 for dim in value["shape"]),"tensor shape非法")
        offsets=value.get("data_offsets")
        require(type(offsets) is list and len(offsets)==2 and
                all(type(x) is int for x in offsets) and 0<=offsets[0]<=offsets[1]<=data_len,
                "tensor data offsets越界")
        names.append(key)
    require(bool(names),"safetensors文件没有tensor")
    return {"tensors":len(names),"header_sha256":hashlib.sha256(head).hexdigest(),
            "header_size":n,"file_size":size,"data_size":data_len,
            "tensor_names_digest":digest(sorted(names)),"_tensor_names":names}


def _no_duplicate_object(items):
    out={}
    for k,v in items:
        require(k not in out,"重复safetensors JSON字段")
        out[k]=v
    return out


def inspect_local_checkpoint(path: str|Path, *, full_weight_hash=False,
                             max_weight_bytes: int=2*1024**3) -> dict:
    """metadata仅辅助检查文件身份；默认并不声称哈希全部权重。"""
    require(type(full_weight_hash) is bool,"full_weight_hash必须是bool")
    require(type(max_weight_bytes) is int and max_weight_bytes>0,"资源上限需要正整数")
    folder=ensure_local_directory(path)
    cfg=read_limited_json(folder/"config.json")
    require(type(cfg) is dict and type(cfg.get("model_type")) is str,
            "config.json缺少model_type；不会猜测模型架构")
    require(cfg.get("is_encoder_decoder",False) is False,"仅支持decoder-only因果LM")
    shards=sorted(folder.glob("*.safetensors"))
    require(bool(shards),"没有safetensors；本课拒绝回退到pickle权重加载")
    require(all(p.is_file() and not p.is_symlink() for p in shards),"权重分片必须是普通文件")
    index_files=sorted(folder.glob("*.safetensors.index.json"))
    require(len(index_files)<=1,"多个safetensors索引无法确认完整权重")
    require(len(shards)==1 or bool(index_files),
            "多个safetensors分片需要明确的weight_map索引")
    if index_files:
        idx=read_limited_json(index_files[0],max_bytes=8*2**20)
        require(type(idx) is dict and type(idx.get("weight_map")) is dict and bool(idx["weight_map"]),
                "权重索引缺少weight_map")
        listed={str(v) for v in idx["weight_map"].values()}
        require(all(type(v) is str and v==Path(v).name and v.endswith(".safetensors") for v in listed),
                "索引权重文件不能穿越目录")
        require(listed=={p.name for p in shards},"索引和权重分片集合不一致")
    total=sum(p.stat().st_size for p in shards)
    require(total<=max_weight_bytes,
            "safetensors总字节超过显式本地资源闸门；没有尝试载入模型")
    infos=[]
    shard_names={}
    for p in shards:
        header=safetensors_header(p)
        shard_names[p.name]=set(header.pop("_tensor_names"))
        infos.append({"file":p.name,"bytes":p.stat().st_size,
                      "mtime_ns":p.stat().st_mtime_ns,"header":header})
    if index_files:
        mapping=idx["weight_map"]
        require(set(mapping)==set().union(*shard_names.values()),
                "weight_map与各safetensors header的tensor集合不匹配")
        require(all(tensor in shard_names[file_name] for tensor,file_name in mapping.items()),
                "weight_map把tensor映射到了错误的shard")
    if full_weight_hash:
        for part,p in zip(infos,shards):part["full_sha256"]=stream_sha256(p)
    small=[]
    for name in ("config.json","tokenizer.json","tokenizer_config.json","special_tokens_map.json",
                 "generation_config.json","chat_template.jinja","tokenizer.model","spiece.model",
                 "sentencepiece.bpe.model","vocab.json","vocab.txt","merges.txt","added_tokens.json",
                 *(p.name for p in index_files)):
        p=folder/name
        if p.exists():
            require(p.is_file() and not p.is_symlink(),"辅助文件不能是符号链接")
            require(p.stat().st_size<=64*2**20,"配置/tokenizer文件超出本课审计上限")
            small.append({"file":name,"bytes":p.stat().st_size,"sha256":stream_sha256(p),
                          "mtime_ns":p.stat().st_mtime_ns})
    require(bool(small) and small[0]["file"]=="config.json","必须审计config.json")
    identity_material={"shards":infos,"small_files":small,"model_type":cfg["model_type"]}
    return {"scope":"local_files_metadata_only" if not full_weight_hash else "local_full_weight_sha256",
            "directory_label":folder.name,"model_type":cfg["model_type"],
            "weight_file_count":len(shards),"weight_total_bytes":total,
            "weight_mib":round(total/2**20,3),
            "max_weight_bytes_gate":max_weight_bytes,
            "metadata_fingerprint":digest(identity_material),
            "full_weight_content_hashed":full_weight_hash,
            "cryptographic_weight_identity":full_weight_hash,
            "weight_files":infos,"small_files":small,
            "warning":"metadata+header fingerprint is not full tensor identity" if not full_weight_hash
                      else "full local weights hashed; still not a trust/security certificate"}


def validate_output_outside_model(model_dir: Path,output_root: str|Path):
    folder=ensure_local_directory(model_dir)
    out=Path(output_root).expanduser().resolve()
    require(out!=folder and folder not in out.parents,
            "报告目录不能位于模型目录内部，避免修改源checkpoint")

### 两份文件：元数据不变，但载荷不同

为方便离线练习，下列自制的最小 Safetensors 文件仅有一个 F32 张量。我们修改它的值，同时恢复文件修改时间与字节长度，来演示为什么只靠文件大小和 mtime 不够。

这份 fixture 是静态文件格式测试，不是可运行的完整 HF 模型。

In [19]:
def fake_safetensors_bytes(value: float=0.25) -> bytes:
    """用于协议测试的合法单张量文件，不是真实LM权重，不需额外依赖。"""
    header={"toy.weight":{"dtype":"F32","shape":[1],"data_offsets":[0,4]}}
    raw=canonical(header).encode("utf-8")
    raw+=b" "*((8-len(raw)%8)%8)
    return struct.pack("<Q",len(raw))+raw+struct.pack("<f",float(value))


def make_fake_local_checkpoint(folder: Path) -> None:
    folder.mkdir(parents=True,exist_ok=True)
    write_json(folder/"config.json",{"model_type":"toy_mock","is_encoder_decoder":False,
                                      "max_position_embeddings":256})
    (folder/"model.safetensors").write_bytes(fake_safetensors_bytes())
    write_json(folder/"tokenizer_config.json",{"chat_template":"mock_only"})

In [20]:
with tempfile.TemporaryDirectory() as td:
    local=Path(td)/'fixture_checkpoint'
    make_fake_local_checkpoint(local)
    before_metadata=inspect_local_checkpoint(local,max_weight_bytes=100000)
    before_full=inspect_local_checkpoint(local,full_weight_hash=True,max_weight_bytes=100000)
    file=local/'model.safetensors'
    oldstat=file.stat()
    file.write_bytes(fake_safetensors_bytes(value=0.50))
    os.utime(file,ns=(oldstat.st_atime_ns,oldstat.st_mtime_ns))
    after_metadata=inspect_local_checkpoint(local,max_weight_bytes=100000)
    after_full=inspect_local_checkpoint(local,full_weight_hash=True,max_weight_bytes=100000)
    print('仅元数据模式发现变化?',before_metadata['metadata_fingerprint']!=after_metadata['metadata_fingerprint'])
    print('完整权重哈希模式发现变化?',before_full['metadata_fingerprint']!=after_full['metadata_fingerprint'])

仅元数据模式发现变化? False
完整权重哈希模式发现变化? True


## 10. 真实本地 HF 评测：明确手动开启，默认不运行

运行条件：你已获得**可信、完整、兼容的本地纯文本 CausalLM Safetensors checkpoint**，并另外安装兼容的 Transformers。下面接口不下载权重：`local_files_only=True`、`trust_remote_code=False`、`use_safetensors=True`。

只用官方 tokenizer 的 Chat Template，不猜测 Qwen/Llama/Gemma 的特殊 ID；配置中的多个 EOS 会全部传入生成器。每次 `batch=1`、贪心、`num_beams=1`，验证返回序列仍包含完全相同的提示前缀，才切出新增 token 并评分。

如果 tokenizer 和模型没有可用 EOS、Chat Template 或明确的上下文上限，**直接报错**，不伪装为低分。若 PAD 与 EOS 别名，报告会显式标出，无法分辨的尾部填充返回未知。

静态前置检查不是恶意模型沙箱，`trust_remote_code=False` 不等于文件绝对可信。不要为“跑通”移除检查。只读是**不调用训练、不预期修改 checkpoint**的实验约定；并非文件系统沙箱。

In [21]:
def _import_hf_model_classes():
    try:
        from transformers import AutoTokenizer,AutoModelForCausalLM
    except ImportError as exc:
        raise LessonError("真实本地模式需要用户另行安装兼容Transformers；离线toy无需它") from exc
    return AutoTokenizer,AutoModelForCausalLM


def _hf_prefix(tokenizer,messages: list[dict]) -> list[int]:
    require(getattr(tokenizer,"chat_template",None) is not None,
            "tokenizer未提供chat_template，不能猜测对话边界")
    prefix=tokenizer.apply_chat_template(messages,tokenize=True,add_generation_prompt=True)
    if isinstance(prefix,torch.Tensor):prefix=prefix.tolist()
    require(type(prefix) is list and bool(prefix) and all(type(i) is int and i>=0 for i in prefix),
            "Chat Template必须返回一维非空token IDs")
    return prefix


def evaluate_local_hf_readonly(path: str|Path, output_root: str|Path, *,device_name="cpu",
                              max_new_tokens=32, num_cases=4,max_weight_bytes=2*1024**3,
                              full_weight_hash=False) -> dict:
    require(type(max_new_tokens) is int and 1<=max_new_tokens<=128,"真实模型生成预算需要1..128")
    require(type(num_cases) is int and 1<=num_cases<=len(fixed_cases()),"num_cases超出固定集合")
    model_dir=ensure_local_directory(path)
    validate_output_outside_model(model_dir,output_root)
    before=inspect_local_checkpoint(model_dir,full_weight_hash=full_weight_hash,
                                    max_weight_bytes=max_weight_bytes)
    device=device_from_name(device_name)
    AutoTokenizer,AutoModelForCausalLM=_import_hf_model_classes()
    tokenizer=AutoTokenizer.from_pretrained(str(model_dir),local_files_only=True,
                                            trust_remote_code=False)
    require(getattr(tokenizer,"chat_template",None) is not None,"模型缺少可用chat_template")
    dtype=torch.float16 if device.type=="cuda" else torch.float32
    model=AutoModelForCausalLM.from_pretrained(str(model_dir),local_files_only=True,
               trust_remote_code=False,use_safetensors=True,dtype=dtype)
    require(not bool(getattr(model.config,"is_encoder_decoder",False)),"只支持decoder-only文本CausalLM")
    model.to(device).eval().requires_grad_(False)
    eos_value=getattr(getattr(model,"generation_config",None),"eos_token_id",None)
    if eos_value is None:eos_value=getattr(tokenizer,"eos_token_id",None)
    eos_ids=normalize_stop_ids(eos_value,len(tokenizer))
    pad_id=getattr(tokenizer,"pad_token_id",None)
    if pad_id is None:pad_id=getattr(getattr(model,"generation_config",None),"pad_token_id",None)
    if pad_id is None:pad_id=eos_ids[0]
    max_context=getattr(model.config,"max_position_embeddings",None)
    require(type(max_context) is int and max_context>0,
            "无法从config确认上下文上限；本课不猜测可用长度")
    require(all(not p.requires_grad for p in model.parameters()),"本地模型必须只读冻结")
    rows=[]
    for c in fixed_cases()[:num_cases]:
        prefix=_hf_prefix(tokenizer,messages_for_task(c["question"]))
        require(len(prefix)+max_new_tokens<=max_context,"提示加生成预算超出模型上下文")
        ids=torch.tensor([prefix],device=device,dtype=torch.long)
        with torch.inference_mode():
            # batch=1、greedy、num_beams=1；本课不评估采样概率或批间padding。
            returned=model.generate(input_ids=ids,attention_mask=torch.ones_like(ids),
                       do_sample=False,num_beams=1,num_return_sequences=1,
                       max_new_tokens=max_new_tokens,eos_token_id=eos_ids,pad_token_id=pad_id,
                       return_dict_in_generate=True,output_scores=False)
        sequences=returned.sequences
        require(isinstance(sequences,torch.Tensor) and sequences.ndim==2 and sequences.shape[0]==1,
                "generate结果需要batch=1的完整序列")
        actual=sequences[0].tolist()
        require(actual[:len(prefix)]==prefix,"generate修改/省略了输入前缀，不能继续评分")
        generated=actual[len(prefix):]
        result=grade_generation(tokenizer,generated,eos_ids,max_new_tokens,c["expected"],pad_id=pad_id)
        rows.append({"id":c["id"],"expected":c["expected"],
                     "prompt_digest":digest(prefix),"prompt_len":len(prefix),
                     "generated_token_ids":generated,"generated_token_count":len(generated),**result})
    after=inspect_local_checkpoint(model_dir,full_weight_hash=full_weight_hash,
                                   max_weight_bytes=max_weight_bytes)
    require(before["metadata_fingerprint"]==after["metadata_fingerprint"],
            "checkpoint文件在只读评测前后发生变化")
    return {"scope":"actual_local_hf_checkpoint_evaluated","tokenizer_class":tokenizer.__class__.__name__,
            "model_class":model.__class__.__name__,"torch":str(torch.__version__),
            "device":str(device),"generation":{"do_sample":False,"num_beams":1,
                        "max_new_tokens":max_new_tokens,"eos_ids":eos_ids,"pad_id":pad_id,
                        "pad_alias_eos":pad_id in eos_ids,"batch_size":1},
            "checkpoint_identity":before,"unchanged_metadata_fingerprint":True,
            "summary":summarize_results(rows),"rows":rows,
            "limitations":["greedy_single_sample_only","no_PEFT_or_training",
                 "full_weight_identity_requires_full_weight_hash_flag",
                 "source_files_not_modified_but_not_a_malicious_code_sandbox"]}

In [22]:
LOCAL_MODEL_DIR = None    # 示例：'/trusted/path/to/local/safetensors_model'
RUN_LOCAL_INSPECT = False
RUN_LOCAL_INFERENCE = False  # 需要上述目录和已安装的Transformers
LOCAL_MAX_WEIGHT_BYTES = 2 * 1024**3
LOCAL_FULL_WEIGHT_HASH = False
if LOCAL_MODEL_DIR is not None and RUN_LOCAL_INSPECT:
    info=inspect_local_checkpoint(LOCAL_MODEL_DIR,full_weight_hash=LOCAL_FULL_WEIGHT_HASH,
                                   max_weight_bytes=LOCAL_MAX_WEIGHT_BYTES)
    print('静态本地目录检查：',info['model_type'],info['scope'])
if LOCAL_MODEL_DIR is not None and RUN_LOCAL_INFERENCE:
    local_report=evaluate_local_hf_readonly(
        LOCAL_MODEL_DIR,'lesson_68_outputs',device_name='cpu',num_cases=4,
        max_weight_bytes=LOCAL_MAX_WEIGHT_BYTES,full_weight_hash=LOCAL_FULL_WEIGHT_HASH)
    print('真实本地评测（只有显式选择才会运行）：',local_report['summary'])
else:
    print('未运行真实HF模型（这是默认行为）。')

未运行真实HF模型（这是默认行为）。


## 11. 完整默认离线运行与文件归档

每次新建时间戳目录并写出本课随机 checkpoint，完成真正的 greedy 与固定随机采样、独立对象重载、文件 SHA、运行张量摘要和人工夹具验证。

输出目录中的文件包括：

| 文件 | 内容 |
|---|---|
| `toy_random_model.pt` | 本课生成的随机微型模型权重；只信任自己生成的文件 |
| `report.json` | 环境、模型身份、分母、停止状态及不可外推的局限 |
| `greedy_rows.json` | 全部 toy 贪心生成 IDs、解码、EOS 与评分 |
| `sampled_rows.json` | 固定种子采样的全部真实 toy 输出 |
| `fixture_rows.json` | 明确标注为脚本式正反例，不是真实模型输出 |
| `cases.json` / `contract.json` | 自建题目、模板、数据摘要和版本 |
| `checks.json` | 独立固定测试清单 |

全部指标来自实际执行，没有提前填入准确率。跨设备、框架版本的随机结果不承诺逐位相同；同环境本次从独立对象重载应一致。

In [23]:
def new_output_dir(root: Path) -> Path:
    root=Path(root)
    name=datetime.now(timezone.utc).strftime("run_%Y%m%dT%H%M%S_%fZ")
    out=root/name
    out.mkdir(parents=True,exist_ok=False)
    return out


def run_offline_audit(output_root="lesson_68_outputs", *,seed=6801,limit=32) -> tuple[Path,dict]:
    require(type(limit) is int and 1<=limit<=64,"toy生成预算非法")
    torch.set_num_threads(1)
    out=new_output_dir(Path(output_root))
    cases=fixed_cases()
    checkpoint=create_toy_checkpoint(out,seed)
    path=toy_checkpoint_path(out)
    on_disk_before=stream_sha256(path)
    model,tok,identity=load_toy_checkpoint(path,"cpu")
    first=evaluate_toy_readonly(model,tok,cases,max_new_tokens=limit,greedy=True)
    second=evaluate_toy_readonly(model,tok,cases,max_new_tokens=limit,greedy=False,seed=6809)
    # 第二套完全独立对象复现同一生成，不能只依靠同一模型实例重跑。
    model2,tok2,identity2=load_toy_checkpoint(path,"cpu")
    again=evaluate_toy_readonly(model2,tok2,cases,max_new_tokens=limit,greedy=False,seed=6809)
    require(second["rows"]==again["rows"],"全新模型重载后的固定采样不一致")
    require(on_disk_before==stream_sha256(path),"只读过程改变了toy模型文件")
    require(identity==identity2,"toy模型重载身份不一致")
    scripted=scripted_fixture_results()
    report={"version":VERSION,"mode":"toy_offline_random_model_not_pretrained",
            "environment":{"python":platform.python_version(),"torch":str(torch.__version__),
                           "device":"cpu","threads":1,"cuda_tested":False},
            "checkpoint":checkpoint,"identity":identity,"dataset":{"cases":len(cases),
                   "sha256":digest(cases),"all_synthetic":True},
            "generation":{"max_new_tokens":limit,"sample_seed":6809,"greedy":True,
                           "sampling":"torch_multinomial_seeded_no_topk_or_temperature",
                           "eos_ids":[tok.eos_token_id],"pad_id":tok.pad_token_id},
            "greedy_summary":first["summary"],"sampled_summary":second["summary"],
            "independent_reload_identical":True,"model_state_unchanged":True,
            "checkpoint_file_unchanged":True,
            "scripted_fixture_summary":scripted["summary"],
            "boundaries":["toy_random_weights_not_pretrained","no_model_training",
                 "exact_JSON_and_EOS_only","no_claim_of_real_HF_checkpoint_run",
                 "optional_local_HF_requires_user_action_and_compatible_dependencies"]}
    write_json(out/"report.json",report)
    write_json(out/"greedy_rows.json",first["rows"])
    write_json(out/"sampled_rows.json",second["rows"])
    write_json(out/"fixture_rows.json",scripted["rows"])
    write_json(out/"cases.json",cases)
    write_json(out/"contract.json",{"version":VERSION,"tokenizer":tok.contract(),
                                   "grading":{"strict_json":True,"requires_eos":True},
                                   "sources":SOURCES})
    print("生成toy审计目录：",out.resolve())
    print("真实checkpoint已加载评测？否；本次仅课堂随机toy权重")
    print("toy greedy：",first["summary"]["successes"],"/",len(cases))
    print("toy sampled：",second["summary"]["successes"],"/",len(cases))
    return out,report

In [24]:
RUN_DEFAULT_AUDIT = True
OUTPUT_DIR = None
REPORT = None
if RUN_DEFAULT_AUDIT:
    OUTPUT_DIR,REPORT = run_offline_audit('lesson_68_outputs',seed=6801,limit=32)
    print('实际随机toy模型：',REPORT['greedy_summary'],REPORT['sampled_summary'])
else:
    print('本次离线审计被显式跳过；不展示虚构结果。')

生成toy审计目录： /mnt/data/_lesson68_build/notebook_runs/lesson_68_outputs/run_20261103T143318_496267Z
真实checkpoint已加载评测？否；本次仅课堂随机toy权重
toy greedy： 0 / 8
toy sampled： 0 / 8
实际随机toy模型： {'total': 8, 'eligible': 8, 'unknown': 0, 'successes': 0, 'strict_success_over_eligible': 0.0, 'strict_success_over_all': 0.0, 'states': {'eos': 0, 'budget_cap': 8, 'interrupted': 0, 'ambiguous_stop_padding': 0}, 'grade_reasons': {'missing_eos': 8}, 'denominator_policy': 'unknown excludes infrastructure interruption and pad/EOS ambiguity; also report over_all'} {'total': 8, 'eligible': 8, 'unknown': 0, 'successes': 0, 'strict_success_over_eligible': 0.0, 'strict_success_over_all': 0.0, 'states': {'eos': 1, 'budget_cap': 7, 'interrupted': 0, 'ambiguous_stop_padding': 0}, 'grade_reasons': {'decode_error': 1, 'missing_eos': 7}, 'denominator_policy': 'unknown excludes infrastructure interruption and pad/EOS ambiguity; also report over_all'}


## 12. 自动验证：不仅检查模型会返回字符串

固定检查覆盖 Chat Template、特殊token污染、EOS/多EOS、预算、PAD/EOS 别名、JSON 严格解析、未知分母、UTF-8、toy模型参数冻结与重载、Safetensors header、静态身份等级、模型目录防覆写，以及**mock HF接口接线**。

mock 中即使返回合格 JSON，也只是验证参数传递和评测链路，不能声称运行过真实预训练权重。真正 HF 模型必须由用户另行指定真实本地 checkpoint。

下面代码比较长，是可展开的逐项断言。先定义练习检查器与参考实现，然后运行固定测试。

In [25]:
class Checks:
    def __init__(self):self.names=[]
    def ok(self,name,condition):
        if not bool(condition):raise AssertionError(name)
        self.names.append(name)
    def eq(self,name,x,y):self.ok(name,x==y)
    def close(self,name,x,y,tol=1e-7):
        self.ok(name,torch.allclose(torch.as_tensor(x,dtype=torch.float64),
                                     torch.as_tensor(y,dtype=torch.float64),atol=tol,rtol=tol))
    def raises(self,name,fn,kinds=(LessonError,ValueError,TypeError,OSError,UnicodeError)):
        try:fn()
        except kinds:self.names.append(name);return
        raise AssertionError(name+": 应拒绝但未拒绝")


def check_exercise_a(fn):
    eos=[2,4]
    c=Checks()
    c.eq("A_正常EOS",fn([10,4],eos,3),"eos")
    c.eq("A_EOS优先于恰达预算",fn([10,11,2],eos,3),"eos")
    c.eq("A_预算已满但无EOS",fn([10,11,12],eos,3),"budget_cap")
    c.eq("A_提前返回无EOS不应当成功",fn([10,11],eos,3),"interrupted")
    c.raises("A_EOS后有非PAD必须拒绝",lambda:fn([2,10],eos,3))
    return {"passed":len(c.names),"names":c.names}


def check_exercise_b(fn):
    c=Checks()
    c.eq("B_精确JSON成功",fn('{"answer":"5"}',"eos","5"),1.0)
    for text in ('{"answer":5}','{"answer":"5","other":1}',
                 '{"answer":"5","answer":"5"}','{"answer":"6"}'):
        c.eq("B_严格结构或值_"+text,fn(text,"eos","5"),0.0)
    c.eq("B_无EOS即使正文正确也0",fn('{"answer":"5"}',"budget_cap","5"),0.0)
    c.eq("B_基础设施中断返回未知",fn('{"answer":"5"}',"interrupted","5"),None)
    return {"passed":len(c.names),"names":c.names}


def check_exercise_c(fn):
    c=Checks()
    sample=[{"grade":{"eligible":True,"reward":1.}},
            {"grade":{"eligible":True,"reward":0.}},
            {"grade":{"eligible":False,"reward":None}}]
    got=fn(sample)
    c.ok("C_分母排除未知",isinstance(got,dict) and got["eligible"]==2)
    c.eq("C_未知计数",got["unknown"],1)
    c.close("C_合格任务成功率",got["strict_success_over_eligible"],.5)
    c.eq("C_全部未知时不能伪称0成功率",fn(sample[2:])["strict_success_over_eligible"],None)
    return {"passed":len(c.names),"names":c.names}

## 13. 独立练习 A：停止状态

实现 `stop_state_ex(ids,eos_ids,limit)`，返回 `eos`、`budget_cap`、`interrupted`。必须支持多个结束 ID，**EOS 在预算最后一步仍然算 eos**。EOS 后出现无效正文应拒绝，不能静默截断。

本题只练单样本、没有 padding 的情况；完整工程函数在上文。练习开关默认 False。

In [26]:
def stop_state_ex(ids, eos_ids, limit):
    raise NotImplementedError('请完成停止状态判定')

RUN_EXERCISE_A = False
if RUN_EXERCISE_A:
    print(check_exercise_a(stop_state_ex))

## 14. 独立练习 B：严格 JSON 与 EOS 奖励

实现 `grade_json_ex(body,state,expected)`：合格 JSON 对象必须只有字符串字段 `answer`，且值与预期完全一致。EOS 条件下成功=1，格式/值不符=0；到预算但没有 EOS=0；基础设施中断返回 `None`。

不要通过 `json.loads()` 后直接 `obj.get('answer')` 忽略重复键、字段类型与多余字段。

In [27]:
def grade_json_ex(body, state, expected):
    raise NotImplementedError('请完成严格JSON奖励')

RUN_EXERCISE_B = False
if RUN_EXERCISE_B:
    print(check_exercise_b(grade_json_ex))

## 15. 独立练习 C：正确汇总合格分母

实现 `coverage_ex(rows)`：每行包含 `grade` 对象，`eligible` 为布尔值，`reward` 为 1/0 或未知 `None`。

返回：合格样本数、未知样本数、成功数与 `strict_success_over_eligible`。不能把未知默默算成 0。若合格分母为零，应返回 `None` 而不是数值 0 或除零错误。

In [28]:
def coverage_ex(rows):
    raise NotImplementedError('请完成可核验的合格分母汇总')

RUN_EXERCISE_C = False
if RUN_EXERCISE_C:
    print(check_exercise_c(coverage_ex))

## 16. 独立参考答案

先练再看。参考实现使用独立名字，不覆盖你的 `*_ex` 函数。完整工程还需要输入、资源与身份检查，这些不应被简化的练习答案替代。

In [29]:
def answer_stop_state(ids: list[int], eos_ids: list[int], limit: int):
    result=audit_generated_tokens(ids,eos_ids,limit)
    return result["state"]


def answer_strict_grade(body: str, state: str, expected: str):
    return grade_strict_json(body,state,expected)["reward"]


def answer_coverage(rows: list[dict]):
    require(bool(rows),"需要非空评测结果")
    valid=[r for r in rows if r["grade"]["eligible"]]
    successes=sum(r["grade"]["reward"] for r in valid)
    return {"eligible":len(valid),"unknown":len(rows)-len(valid),
            "strict_success_over_eligible":None if not valid else successes/len(valid)}

In [30]:
print('参考A:',check_exercise_a(answer_stop_state))
print('参考B:',check_exercise_b(answer_strict_grade))
print('参考C:',check_exercise_c(answer_coverage))

参考A: {'passed': 5, 'names': ['A_正常EOS', 'A_EOS优先于恰达预算', 'A_预算已满但无EOS', 'A_提前返回无EOS不应当成功', 'A_EOS后有非PAD必须拒绝']}
参考B: {'passed': 7, 'names': ['B_精确JSON成功', 'B_严格结构或值_{"answer":5}', 'B_严格结构或值_{"answer":"5","other":1}', 'B_严格结构或值_{"answer":"5","answer":"5"}', 'B_严格结构或值_{"answer":"6"}', 'B_无EOS即使正文正确也0', 'B_基础设施中断返回未知']}
参考C: {'passed': 4, 'names': ['C_分母排除未知', 'C_未知计数', 'C_合格任务成功率', 'C_全部未知时不能伪称0成功率']}


In [31]:
def run_checks() -> dict:
    torch.set_num_threads(1)
    rng=torch.random.get_rng_state().clone()
    torch.manual_seed(6802)
    c=Checks()
    tok=ByteMergeChatTokenizer()
    c.eq("toy词表固定",tok.vocab_size,281)
    c.eq("utf8编码解码往返",tok.decode(tok.encode("你好，世界")),"你好，世界")
    c.eq("合并词元不能靠分段拼接保证",len(tok.encode("OK"))<len(tok.encode("O")+tok.encode("K")),True)
    c.eq("user内容的字面控制标记不是结构token",all(t>=tok.byte_offset for t in tok.encode("<im_start>")),True)
    c.raises("残缺UTF8拒绝自动替换",lambda:tok.decode([tok.byte_offset+0xE4]))
    c.raises("正文含结构EOS拒绝",lambda:tok.decode([tok.eos_token_id]))
    c.raises("未知保留词元拒绝",lambda:tok.decode([8]))
    prefix=tok.apply_chat_template(messages_for_task("2+3"),tokenize=True,add_generation_prompt=True)
    c.eq("toy assistant生成头",prefix[-2:],[tok.im_start_token_id,tok.role_ids["assistant"]])
    c.eq("chat_template不在每条消息中重复插入BOS",prefix.count(tok.bos_token_id),1)
    c.eq("提示包含system和user角色",tok.role_ids["system"] in prefix and tok.role_ids["user"] in prefix,True)
    c.raises("消息伪装角色被拒绝",lambda:tok.apply_chat_template([{"role":"developer","content":"x"}]))
    c.raises("已有assistant又新增assistant头拒绝",lambda:tok.apply_chat_template([{"role":"assistant","content":"hi"}],add_generation_prompt=True))
    c.eq("多停止ID归一化",normalize_stop_ids([2,4]),[2,4])
    c.raises("未配置EOS拒绝",lambda:normalize_stop_ids(None))
    c.raises("EOS重复拒绝",lambda:normalize_stop_ids([2,2]))
    c.raises("EOS越界拒绝",lambda:normalize_stop_ids([281],tok.vocab_size))
    eos=[2,4]
    c.eq("到EOS正常停止",audit_generated_tokens([16,2],eos,5)["state"],"eos")
    c.eq("恰好上限最后EOS仍正确",audit_generated_tokens([16,17,2],eos,3)["state"],"eos")
    c.eq("多EOS配置生效",audit_generated_tokens([16,4],eos,5)["state"],"eos")
    c.eq("没有EOS并到cap",audit_generated_tokens([16,17],eos,2)["state"],"budget_cap")
    c.eq("早于cap无EOS应为中断",audit_generated_tokens([16],eos,3)["state"],"interrupted")
    c.eq("明确基础设施中断不算任务0分",audit_generated_tokens([16,2],eos,3,interrupted=True)["eligible"],False)
    c.eq("尾部独立PAD可忽略",audit_generated_tokens([16,2,0,0],eos,5,pad_id=0)["state"],"eos")
    c.eq("EOS和PAD别名且尾部填充状态不确定",audit_generated_tokens([16,2,2],eos,5,pad_id=2)["state"],"ambiguous_stop_padding")
    c.raises("停止后普通token拒绝",lambda:audit_generated_tokens([16,2,17],eos,5,pad_id=0))
    c.raises("无PAD配置时停止后还有token拒绝",lambda:audit_generated_tokens([16,2,0],eos,5))
    c.raises("生成超过预算拒绝",lambda:audit_generated_tokens([16,17,18],eos,2))
    c.raises("生成token类型拒绝",lambda:audit_generated_tokens([16,True],eos,3))
    c.raises("生成预算零拒绝",lambda:audit_generated_tokens([16],eos,0))
    c.eq("严格JSON正确奖励",grade_strict_json('{"answer":"5"}',"eos","5")["reward"],1.)
    c.eq("字面数字不等于JSON",grade_strict_json('5',"eos","5")["reward"],0.)
    c.eq("JSON数字不能冒充字符串答案",grade_strict_json('{"answer":5}',"eos","5")["reward"],0.)
    c.eq("JSON多余字段不允许",grade_strict_json('{"answer":"5","q":1}',"eos","5")["reward"],0.)
    c.eq("JSON重复字段不允许",grade_strict_json('{"answer":"5","answer":"5"}',"eos","5")["reward"],0.)
    c.eq("JSON非法常量不接受",grade_strict_json('{"answer":"5","x":NaN}',"eos","5")["reward"],0.)
    c.eq("答案错误结果为0",grade_strict_json('{"answer":"6"}',"eos","5")["reward"],0.)
    c.eq("预算结束时即使正文正确也为0",grade_strict_json('{"answer":"5"}',"budget_cap","5")["reward"],0.)
    c.eq("中断不产生0奖励",grade_strict_json(None,"interrupted","5")["reward"],None)
    c.eq("padEOS不明确时不评分",grade_strict_json(None,"ambiguous_stop_padding","5")["eligible"],False)
    c.eq("解码错误时EOS完整算格式无效",grade_strict_json(None,"eos","5",decode_error="UnicodeDecodeError")["reward"],0.)
    c.raises("未知停止状态拒绝",lambda:grade_strict_json(None,"unknown","5"))
    c.raises("非法expected拒绝",lambda:grade_strict_json('{"answer":"5"}',"eos","five"))
    c.eq("保留不完整字节并记录解码错误",grade_generation(tok,[tok.byte_offset+0xE4,tok.eos_token_id],[tok.eos_token_id],3,"5")["grade"]["reason"],"decode_error")
    cases=fixed_cases()
    c.eq("固定8题",len(cases),8)
    c.eq("数据可重复生成",digest(cases),digest(fixed_cases()))
    bad=copy.deepcopy(cases);bad[1]["id"]=bad[0]["id"]
    c.raises("题号重复拒绝",lambda:validate_cases(bad))
    c.raises("非法题号拒绝",lambda:validate_cases([{"id":"../x","question":"hi","expected":"1"}]))
    fixture=scripted_fixture_results()
    by_name={r["fixture"]:r for r in fixture["rows"]}
    c.eq("脚本式正例必须确切EOS",by_name["correct_eos"]["grade"]["reward"],1.)
    c.eq("脚本式错误值为0",by_name["wrong_eos"]["grade"]["reward"],0.)
    c.eq("脚本式格式错误为0",by_name["schema_eos"]["grade"]["reward"],0.)
    c.eq("脚本式cap为0",by_name["budget_cap"]["grade"]["reward"],0.)
    c.eq("脚本式提前中断为未知",by_name["interrupted_early"]["grade"]["reward"],None)
    c.eq("脚本式显式中断为未知",by_name["explicit_interrupted"]["grade"]["reward"],None)
    c.eq("脚本式UTF8残缺被保留",by_name["invalid_utf8_eos"]["grade"]["reason"],"decode_error")
    c.eq("汇总分母不混未知",fixture["summary"]["eligible"],5)
    c.eq("汇总未知计数",fixture["summary"]["unknown"],2)
    c.close("汇总仅合格答案成功率",fixture["summary"]["strict_success_over_eligible"],.2)
    c.eq("全部未知结果的成功率为None",summarize_results([by_name["explicit_interrupted"]])["strict_success_over_eligible"],None)
    c.eq("参考练习A通过",check_exercise_a(answer_stop_state)["passed"],5)
    c.eq("参考练习B通过",check_exercise_b(answer_strict_grade)["passed"],7)
    c.eq("参考练习C通过",check_exercise_c(answer_coverage)["passed"],4)
    with tempfile.TemporaryDirectory() as temp:
        folder=Path(temp)
        ck=create_toy_checkpoint(folder/"toy",seed=6801)
        c.eq("toy权重真实落盘",(folder/"toy"/"toy_random_model.pt").is_file(),True)
        m,t,id1=load_toy_checkpoint(folder/"toy"/"toy_random_model.pt")
        c.eq("toy检查点是随机非预训练",id1["scope"],"toy_random_not_pretrained")
        wrong=torch.load(folder/"toy"/"toy_random_model.pt",weights_only=True)
        with torch.no_grad():wrong["state"]["token.weight"][0,0]+=0.125
        torch.save(wrong,folder/"tampered_toy.pt")
        c.raises("toy张量被修改但摘要不更新时拒绝",lambda:load_toy_checkpoint(folder/"tampered_toy.pt"))
        c.eq("tensor状态hash可重复",tensor_state_hash(m.state_dict()),ck["state_hash"])
        c.eq("所有toy参数冻结",all(not p.requires_grad for p in m.parameters()),True)
        small=cases[:2]
        r=evaluate_toy_readonly(m,t,small,max_new_tokens=8,seed=6801,greedy=True)
        c.eq("真正模型推理前后权重不变",r["before_hash"],r["after_hash"])
        c.eq("真正模型推理不产生梯度",all(p.grad is None for p in m.parameters()),True)
        c.eq("toy生成固定预算",all(len(x["generated_token_ids"])<=8 for x in r["rows"]),True)
        m2,t2,_=load_toy_checkpoint(folder/"toy"/"toy_random_model.pt")
        c.eq("新对象toy贪心输出一致",evaluate_toy_readonly(m2,t2,small,8,greedy=True)["rows"],r["rows"])
        a=evaluate_toy_readonly(m,t,small,8,greedy=False,seed=610)
        b=evaluate_toy_readonly(m2,t2,small,8,greedy=False,seed=610)
        c.eq("固定种子采样可从新对象复现",a["rows"],b["rows"])
        c.eq("toy文件只读评测后内容未变",stream_sha256(folder/"toy"/"toy_random_model.pt"),ck["file_sha256"])
        c.raises("toy超上下文拒绝",lambda:generate_toy_ids(m,[1]*245,t,20))
        c.raises("toy非法生成预算拒绝",lambda:generate_toy_ids(m,[1],t,0))
        c.raises("toy非法device拒绝",lambda:device_from_name("invalid"))
        c.raises("没提供本地目录拒绝",lambda:ensure_local_directory("Org/Online-Model"))
        fake=folder/"local";make_fake_local_checkpoint(fake)
        inspected=inspect_local_checkpoint(fake,max_weight_bytes=100000)
        c.eq("本地静态审计无需transformers",inspected["model_type"],"toy_mock")
        c.eq("默认不冒称完整权重身份",inspected["cryptographic_weight_identity"],False)
        c.eq("full_hash明确计算权重",inspect_local_checkpoint(fake,full_weight_hash=True,max_weight_bytes=100000)["cryptographic_weight_identity"],True)
        c.eq("权重header确实含一个tensor",inspected["weight_files"][0]["header"]["tensors"],1)
        c.raises("资源闸门太低必须拒绝",lambda:inspect_local_checkpoint(fake,max_weight_bytes=5))
        c.raises("报表位置不能在checkpoint内",lambda:validate_output_outside_model(fake,fake/"reports"))
        c.eq("外部输出位置允许",validate_output_outside_model(fake,folder/"reports"),None)
        # header不变、文件大小和mtime恢复时，metadata模式会漏掉payload变化；完整hash可发现。
        weight=fake/"model.safetensors"
        before_metadata=inspect_local_checkpoint(fake,max_weight_bytes=100000)
        before_full=inspect_local_checkpoint(fake,full_weight_hash=True,max_weight_bytes=100000)
        st=weight.stat();weight.write_bytes(fake_safetensors_bytes(value=.5))
        os.utime(weight,ns=(st.st_atime_ns,st.st_mtime_ns))
        after_metadata=inspect_local_checkpoint(fake,max_weight_bytes=100000)
        after_full=inspect_local_checkpoint(fake,full_weight_hash=True,max_weight_bytes=100000)
        c.eq("保留元数据时修改payload不会改metadata fingerprint",
             before_metadata["metadata_fingerprint"],after_metadata["metadata_fingerprint"])
        c.ok("完整内容hash发现payload变化",before_full["metadata_fingerprint"]!=after_full["metadata_fingerprint"])
        bad_header=folder/"bad.safetensors"
        bad_header.write_bytes(struct.pack("<Q",20*2**20)+b"{}")
        c.raises("伪造巨大safetensors header拒绝",lambda:safetensors_header(bad_header))
        bad_header.write_bytes(b"\x00"*12)
        c.raises("损坏权重header拒绝",lambda:safetensors_header(bad_header))
        # 符号链接和pickle回退都不能作为合法本地模型输入。
        only_bin=folder/"pickle_only";only_bin.mkdir()
        write_json(only_bin/"config.json",{"model_type":"toy_mock"})
        (only_bin/"pytorch_model.bin").write_bytes(b"not safe")
        c.raises("pickle-only目录拒绝",lambda:inspect_local_checkpoint(only_bin))
        linked=folder/"link_model"
        try:
            linked.symlink_to(fake,target_is_directory=True)
        except (OSError,NotImplementedError):
            pass
        else:
            c.raises("源模型软链接拒绝",lambda:ensure_local_directory(linked))
        # Mock: 确保可选HF接口调用严格限制为本地、安全配置；不冒称真实模型测试。
        class FakeTokenizer:
            chat_template="mock-only";pad_token_id=0;eos_token_id=2
            def __init__(self):self.inner=ByteMergeChatTokenizer()
            def __len__(self):return self.inner.vocab_size
            def apply_chat_template(self,messages,tokenize=True,add_generation_prompt=True):
                return self.inner.apply_chat_template(messages,tokenize=tokenize,add_generation_prompt=add_generation_prompt)
            def decode(self,ids,skip_special_tokens=False):return self.inner.decode(ids,skip_special_tokens=skip_special_tokens)
            def encode(self,s,add_special_tokens=False):return self.inner.encode(s,add_special_tokens=add_special_tokens)
        class FakeModel(nn.Module):
            def __init__(self):
                super().__init__();self.dummy=nn.Parameter(torch.zeros(1))
                self.config=type("MockConfig",(),{"is_encoder_decoder":False,"max_position_embeddings":256})()
                self.generation_config=type("MockGen",(),{"eos_token_id":[2,4],"pad_token_id":0})()
            def generate(self,**kwargs):
                require(kwargs["do_sample"] is False and kwargs["num_beams"]==1 and
                        kwargs["num_return_sequences"]==1,"不是单样本贪心")
                require(kwargs["eos_token_id"]==[2,4] and kwargs["pad_token_id"]==0,
                        "EOS/pad合同丢失")
                ids=kwargs["input_ids"]
                completion=ByteMergeChatTokenizer().encode('{"answer":"5"}')+[2]
                seq=torch.cat([ids,torch.tensor([completion],dtype=torch.long,device=ids.device)],dim=1)
                return type("Generation",(),{"sequences":seq})()
        class FakeAutoTokenizer:
            @staticmethod
            def from_pretrained(p,**kw):
                require(kw=={"local_files_only":True,"trust_remote_code":False},
                        "tokenizer不可访问远端或执行自定义代码")
                return FakeTokenizer()
        class FakeAutoModel:
            @staticmethod
            def from_pretrained(p,**kw):
                require(kw["local_files_only"] is True and kw["trust_remote_code"] is False and
                        kw["use_safetensors"] is True and kw["dtype"]==torch.float32,
                        "模型加载不符合本地safetensors限制")
                return FakeModel()
        old_import=globals()["_import_hf_model_classes"]
        try:
            globals()["_import_hf_model_classes"]=lambda:(FakeAutoTokenizer,FakeAutoModel)
            mock_eval=evaluate_local_hf_readonly(fake,folder/"mock_output",num_cases=3,
                                                 max_new_tokens=32,max_weight_bytes=100000)
            c.eq("Mock HF评估走完整路径",mock_eval["summary"]["total"],3)
            c.eq("Mock HF返回固定的正确答案数",mock_eval["summary"]["successes"],2)
            c.eq("Mock HF模型源目录保持不变",mock_eval["unchanged_metadata_fingerprint"],True)
        finally:
            globals()["_import_hf_model_classes"]=old_import
    torch.random.set_rng_state(rng)
    return {"count":len(c.names),"names":c.names,
            "scope":"toy随机模型真实推理、模型文件静态审计、mock HF接线；未运行真实HF权重"}

In [32]:
FIXED_CHECKS = run_checks()
print('固定检查通过：',FIXED_CHECKS['count'],'项')
print('检查范围：',FIXED_CHECKS['scope'])
if OUTPUT_DIR is not None:
    write_json(OUTPUT_DIR/'checks.json',FIXED_CHECKS)

固定检查通过： 92 项
检查范围： toy随机模型真实推理、模型文件静态审计、mock HF接线；未运行真实HF权重


## 17. 命令行完成验收

只有自带 PyTorch 的环境：

```bash
python lesson_68_local_checkpoint_readonly_audit.py --checks-only
python lesson_68_local_checkpoint_readonly_audit.py
```

已有本地模型但暂时不安装 HF，仅扫描静态身份：

```bash
python lesson_68_local_checkpoint_readonly_audit.py --inspect-local /trusted/local/model
python lesson_68_local_checkpoint_readonly_audit.py --inspect-local /trusted/local/model --full-weight-hash
```

明确允许真实模型加载、确认 CPU/GPU/内存预算后（默认权重文件闸门2 GiB）：

```bash
python lesson_68_local_checkpoint_readonly_audit.py --local-model /trusted/local/model --device cpu
# 已确认CUDA可用、权重和总显存预算才修改为 --device cuda
```

这里**没有**对本制作环境不存在的真实 HF checkpoint 做能力或显存实测。下限低于硬件预算不保证能加载，GPU性能也不能凭CPU结果猜测。

In [33]:
def main(argv=None) -> int:
    parser=argparse.ArgumentParser(description=__doc__)
    mode=parser.add_mutually_exclusive_group()
    mode.add_argument("--checks-only",action="store_true")
    mode.add_argument("--inspect-local",metavar="MODEL_DIR")
    mode.add_argument("--local-model",metavar="MODEL_DIR")
    parser.add_argument("--device",choices=["cpu","cuda","auto"],default="cpu")
    parser.add_argument("--max-new-tokens",type=int,default=32)
    parser.add_argument("--num-cases",type=int,default=4)
    parser.add_argument("--full-weight-hash",action="store_true")
    parser.add_argument("--max-weight-bytes",type=int,default=2*1024**3)
    parser.add_argument("--output-dir",default="lesson_68_outputs")
    parser.add_argument("--toy-seed",type=int,default=6801)
    args=parser.parse_args(argv)
    torch.set_num_threads(1)
    try:
        if args.checks_only:
            checks=run_checks()
            print("固定检查通过：",checks["count"],"项。真实HF预训练权重未测试。")
            return 0
        if args.inspect_local:
            validate_output_outside_model(args.inspect_local,args.output_dir)
            report=inspect_local_checkpoint(args.inspect_local,full_weight_hash=args.full_weight_hash,
                                            max_weight_bytes=args.max_weight_bytes)
            out=new_output_dir(Path(args.output_dir))
            write_json(out/"local_inventory.json",report)
            print("只读静态审计目录：",out.resolve())
            print("没有加载模型，完整权重哈希？",report["full_weight_content_hashed"])
            return 0
        if args.local_model:
            report=evaluate_local_hf_readonly(args.local_model,args.output_dir,
                device_name=args.device,max_new_tokens=args.max_new_tokens,num_cases=args.num_cases,
                max_weight_bytes=args.max_weight_bytes,full_weight_hash=args.full_weight_hash)
            out=new_output_dir(Path(args.output_dir))
            write_json(out/"actual_local_model_report.json",report)
            print("本地模型只读生成审计：",out.resolve())
            print("任务合格结果：",report["summary"]["successes"],"/",report["summary"]["eligible"])
            return 0
        checks=run_checks()
        out,report=run_offline_audit(args.output_dir,seed=args.toy_seed,limit=args.max_new_tokens)
        write_json(out/"checks.json",checks)
        print("固定检查通过：",checks["count"],"项。")
        return 0
    except (LessonError,OSError,UnicodeError,KeyError) as exc:
        print("评测失败：",type(exc).__name__,str(exc),file=sys.stderr)
        return 2

## 18. 本课验收与下一课

你应能回答：

1. 为什么只看文件名、大小与 mtime 不能证明两份权重相同？
2. 为什么单个生成序列的结束符要看 ID，而不是只看解码后的可见文本？
3. EOS 恰好在预算末尾和没有 EOS 的 cap 有何区别？
4. 为什么 PAD/EOS 别名可能使批量停止的来源不可判定？
5. 为什么基础设施中断不能简单归入失败样本，为什么评分要有明确分母？
6. 为什么 `local_files_only=True`、`trust_remote_code=False` 还不意味着任何本地 checkpoint 都安全？

**下一课（第69课）：真实小模型的 LoRA 微调前预检——训练数据边界、模型参数冻结、适配器身份、梯度与显存安全。** 仍将保持默认离线、独立运行，不把随机 toy 的准确率当真实模型效果。

主要资料：

- [HF Chat Templates：assistant生成头与 token化边界](https://huggingface.co/docs/transformers/chat_templating)
- [HF Generation：EOS列表、max_new_tokens、返回序列](https://huggingface.co/docs/transformers/main_classes/text_generation)
- [HF `from_pretrained` 加载参数](https://huggingface.co/docs/transformers/main_classes/model)
- [Safetensors 文件格式与安全边界](https://huggingface.co/docs/safetensors/index)
- [PyTorch 复现边界](https://docs.pytorch.org/docs/stable/notes/randomness.html)